## Instalación de librerias

In [1]:
import pandas as pd

## Cargar datasets

In [2]:
# Carga los datasets
df_cve = pd.read_csv('../data/processed/cve_data.csv')
df_exploit = pd.read_csv('../data/processed/exploit_data.csv')

print("CVE:", df_cve.shape)
print("Exploit:", df_exploit.shape)

df_cve.head()

CVE: (123802, 5)
Exploit: (27067, 2)


,cve_id,descripcion,fecha,cvss_score,severidad
0,CVE-2023-0028,Cross-site Scripting (XSS) - Stored in GitHub ...,2023-01-01 01:15:12.627,5.7,MEDIUM
1,CVE-2023-0029,A vulnerability was found in Multilaser RE708 ...,2023-01-01 14:15:09.963,5.3,MEDIUM
2,CVE-2023-22551,"The FTP (aka ""Implementation of a simple FTP c...",2023-01-01 18:15:17.157,7.5,HIGH
3,CVE-2023-22451,Kiwi TCMS is an open source test management sy...,2023-01-02 16:15:11.063,6.5,MEDIUM
4,CVE-2023-22452,kenny2automate is a Discord bot. In the web in...,2023-01-02 20:15:10.163,6.5,MEDIUM


## Unir datasets

In [3]:
# Crea indicador de exploit disponible
df_exploit['tiene_exploit'] = 1

# Une por CVE
df = df_cve.merge(df_exploit[['cve_id', 'tiene_exploit']], 
                  on='cve_id', 
                  how='left')

# Rellena nulos (si no tiene exploit -> 0)
df['tiene_exploit'] = df['tiene_exploit'].fillna(0)

df.head()

,cve_id,descripcion,fecha,cvss_score,severidad,tiene_exploit
0,CVE-2023-0028,Cross-site Scripting (XSS) - Stored in GitHub ...,2023-01-01 01:15:12.627,5.7,MEDIUM,0.0
1,CVE-2023-0029,A vulnerability was found in Multilaser RE708 ...,2023-01-01 14:15:09.963,5.3,MEDIUM,0.0
2,CVE-2023-22551,"The FTP (aka ""Implementation of a simple FTP c...",2023-01-01 18:15:17.157,7.5,HIGH,0.0
3,CVE-2023-22451,Kiwi TCMS is an open source test management sy...,2023-01-02 16:15:11.063,6.5,MEDIUM,0.0
4,CVE-2023-22452,kenny2automate is a Discord bot. In the web in...,2023-01-02 20:15:10.163,6.5,MEDIUM,0.0


## Limpieza de datos

In [4]:
# Convierte a tipo numérico
df['cvss_score'] = pd.to_numeric(df['cvss_score'], errors='coerce')

# Elimina los registros sin score
df = df.dropna(subset=['cvss_score'])

df.info()

<class 'pandas.DataFrame'>
Index: 116320 entries, 0 to 123818
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   cve_id         116320 non-null  str    
 1   descripcion    116320 non-null  str    
 2   fecha          116320 non-null  str    
 3   cvss_score     116320 non-null  float64
 4   severidad      116320 non-null  str    
 5   tiene_exploit  116320 non-null  float64
dtypes: float64(2), str(4)
memory usage: 6.2 MB


## Crear variables

### Criticidad CVSS

In [5]:
# Clasificación de severidad
df['criticidad'] = pd.cut(
    df['cvss_score'],
    bins=[0, 4, 7, 9, 10],
    labels=['baja', 'media', 'alta', 'critica']
)

### Antigüedad de vulnerabilidad

In [6]:
# Convierte la fecha
df['fecha'] = pd.to_datetime(df['fecha'])

# Año actual
df['anio_actual'] = 2026

df['antiguedad'] = df['anio_actual'] - df['fecha'].dt.year

### Clasificación tipo OWASP

In [7]:
def clasificar_vuln(desc):
    desc = str(desc).lower()
    
    if 'sql' in desc:
        return 'SQL Injection'
    elif 'xss' in desc:
        return 'XSS'
    elif 'overflow' in desc:
        return 'Buffer Overflow'
    elif 'execute' in desc or 'rce' in desc:
        return 'RCE'
    else:
        return 'Other'

df['categoria'] = df['descripcion'].apply(clasificar_vuln)

## Ranking de riesgo

In [8]:
# Convierte exploit a peso
df['tiene_exploit'] = df['tiene_exploit'].astype(int)

# Score de priorización
df['score_prioridad'] = (
    df['cvss_score'] * 0.7 +
    df['tiene_exploit'] * 3 +
    df['antiguedad'] * 0.3
)

## Ordenar prioridades

In [9]:
# Ordena de mayor a menor riesgo
df = df.sort_values(by='score_prioridad', ascending=False)

df.head(10)

,cve_id,descripcion,fecha,cvss_score,severidad,tiene_exploit,criticidad,anio_actual,antiguedad,categoria,score_prioridad
14967,CVE-2023-3710,Improper Input Validation vulnerability in Hon...,2023-09-12 20:15:09.387,9.9,CRITICAL,1,critica,2026,3,Other,10.83
9202,CVE-2023-36355,TP-Link TL-WR940N V4 was discovered to contain...,2023-06-22 20:15:09.733,9.9,CRITICAL,1,critica,2026,3,Buffer Overflow,10.83
6651,CVE-2023-27823,An authentication bypass in Optoma 1080PSTX C0...,2023-05-12 14:15:09.727,9.8,CRITICAL,1,critica,2026,3,Other,10.76
6650,CVE-2023-1934,"The PnPSCADA system, a product of SDG Technolo...",2023-05-12 14:15:09.653,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76
1140,CVE-2023-23163,Art Gallery Management System Project v1.0 was...,2023-02-10 20:15:53.760,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76
13232,CVE-2023-39115,install/aiz-uploader/upload in Campcodes Onlin...,2023-08-16 15:15:11.113,9.8,CRITICAL,1,critica,2026,3,XSS,10.76
20153,CVE-2023-6019,A command injection existed in Ray's cpu_profi...,2023-11-16 17:15:08.830,9.8,CRITICAL,1,critica,2026,3,RCE,10.76
3129,CVE-2023-27100,Improper restriction of excessive authenticati...,2023-03-22 23:15:12.350,9.8,CRITICAL,1,critica,2026,3,RCE,10.76
2712,CVE-2023-28343,OS command injection affects Altenergy Power C...,2023-03-14 20:15:10.123,9.8,CRITICAL,1,critica,2026,3,Other,10.76
1139,CVE-2023-23162,Art Gallery Management System Project v1.0 was...,2023-02-10 20:15:53.703,9.8,CRITICAL,1,critica,2026,3,SQL Injection,10.76


## Respuesta a preguntas estratégicas/negocio

### 1. ¿Qué porcentaje de vulnerabilidades catalogadas como críticas cuentan realmente con exploit público disponible?

In [10]:
criticas = df[df['criticidad'] == 'critica']

porcentaje = criticas['tiene_exploit'].mean() * 100

print(f"Porcentaje críticas con exploit: {porcentaje:.2f}%")

Porcentaje críticas con exploit: 1.21%


### 2. ¿Existen vulnerabilidades con puntaje medio o bajo que presenten evidencia de explotación activa según el registro Exploit Database?

In [11]:
bajas_medias_exploit = df[
    (df['criticidad'].isin(['baja', 'media'])) &  
    (df['tiene_exploit'] == 1)
]

bajas_medias_exploit.head()

,cve_id,descripcion,fecha,cvss_score,severidad,tiene_exploit,criticidad,anio_actual,antiguedad,categoria,score_prioridad
9169,CVE-2023-34927,Casdoor v1.331.0 and below was discovered to c...,2023-06-22 13:15:10.383,6.5,MEDIUM,1,media,2026,3,Other,8.45
9168,CVE-2023-34927,Casdoor v1.331.0 and below was discovered to c...,2023-06-22 13:15:10.383,6.5,MEDIUM,1,media,2026,3,Other,8.45
8280,CVE-2023-32750,Pydio Cells through 4.1.2 allows SSRF. For lon...,2023-06-08 21:15:17.340,6.5,MEDIUM,1,media,2026,3,Other,8.45
9170,CVE-2023-34927,Casdoor v1.331.0 and below was discovered to c...,2023-06-22 13:15:10.383,6.5,MEDIUM,1,media,2026,3,Other,8.45
8266,CVE-2023-34096,Thruk is a multibackend monitoring webinterfac...,2023-06-08 19:15:09.773,6.5,MEDIUM,1,media,2026,3,Other,8.45


### 3. ¿Qué categorías de vulnerabilidad concentran mayor frecuencia o mayor disponibilidad de exploits?

In [12]:
df['categoria'].value_counts()

categoria
Other              64212
RCE                22462
XSS                13051
SQL Injection      10030
Buffer Overflow     6565
Name: count, dtype: int64

### 4. ¿Qué criterios permiten construir un ranking alternativo que optimice la asignación de recursos limitados?

In [13]:
print("Variables utilizadas en el modelo:")
print("- CVSS score (severidad técnica)")
print("- Tiene exploit (riesgo real)")
print("- Antigüedad (exposición en el tiempo)")

# Muestra distribución del score
df[['cvss_score', 'tiene_exploit', 'antiguedad', 'score_prioridad']].describe()

Variables utilizadas en el modelo:
- CVSS score (severidad técnica)
- Tiene exploit (riesgo real)
- Antigüedad (exposición en el tiempo)


,cvss_score,tiene_exploit,antiguedad,score_prioridad
count,116320.000000,116320.000000,116320.000000,116320.000000
mean,6.695938,0.004384,1.618269,5.185790
std,1.705486,0.066070,0.922056,1.252659
min,0.000000,0.000000,0.000000,0.000000
25%,5.400000,0.000000,1.000000,4.310000
50%,6.500000,0.000000,2.000000,5.150000
75%,7.800000,0.000000,2.000000,6.060000
max,10.000000,1.000000,3.000000,10.830000


In [14]:
print("\n=== VALORES PROMEDIO (MEDIA) ===")
print(f"CVSS score(promedio): {df['cvss_score'].mean():.2f}")
print(f"Tiene exploit (promedio): {df['tiene_exploit'].mean():.4f}")
print(f"Antigüedad (promedio): {df['antiguedad'].mean():.2f}")


=== VALORES PROMEDIO (MEDIA) ===
CVSS score(promedio): 6.70
Tiene exploit (promedio): 0.0044
Antigüedad (promedio): 1.62


In [15]:
# Importancia de variables (correlación con el score)
df[['score_prioridad', 'cvss_score', 'tiene_exploit', 'antiguedad']].corr()

,score_prioridad,cvss_score,tiene_exploit,antiguedad
score_prioridad,1.000000,0.961193,0.196386,0.239401
cvss_score,0.961193,1.000000,0.032164,0.013853
tiene_exploit,0.196386,0.032164,1.000000,0.033965
antiguedad,0.239401,0.013853,0.033965,1.000000


In [16]:
print("\n=== RELACIÓN CON EL SCORE DE PRIORIDAD ===")
print(f"Score vs CVSS: {df[['score_prioridad','cvss_score']].corr().iloc[0,1]:.2f}")
print(f"Score vs Exploit: {df[['score_prioridad','tiene_exploit']].corr().iloc[0,1]:.2f}")
print(f"Score vs Antigüedad: {df[['score_prioridad','antiguedad']].corr().iloc[0,1]:.2f}")


=== RELACIÓN CON EL SCORE DE PRIORIDAD ===
Score vs CVSS: 0.96
Score vs Exploit: 0.20
Score vs Antigüedad: 0.24


### 5. ¿Qué brechas existen entre la priorización basada únicamente en CVSS y un enfoque basado en integración de datos múltiples?

In [17]:
# Top 10 vulnerabilidades de CVSS
top_cvss = df.sort_values(by='cvss_score', ascending=False).head(10)

# Top 10 vulnerabilidades de este modelo
top_modelo = df.sort_values(by='score_prioridad', ascending=False).head(10)

print("=== TOP 10 SEGÚN CVSS ===")
print(top_cvss[['cve_id', 'cvss_score', 'tiene_exploit']])

print("\n=== TOP 10 SEGÚN MODELO ===")
print(top_modelo[['cve_id', 'cvss_score', 'tiene_exploit']])

# % de vulnerabilidades con exploit en cada ranking
top_cvss_exploit = top_cvss['tiene_exploit'].mean() * 100
top_modelo_exploit = top_modelo['tiene_exploit'].mean() * 100

print("\n=== COMPARACIÓN DE EXPLOITS ===")
print(f"% exploits en Top CVSS: {top_cvss_exploit:.2f}%")
print(f"% exploits en Top Modelo: {top_modelo_exploit:.2f}%")

=== TOP 10 SEGÚN CVSS ===
                cve_id  cvss_score  tiene_exploit
26401   CVE-2023-49815        10.0              0
120931  CVE-2026-31957        10.0              0
44150   CVE-2024-36412        10.0              0
41063   CVE-2024-32700        10.0              0
23776   CVE-2023-48418        10.0              0
59416   CVE-2024-48967        10.0              0
59415   CVE-2024-48966        10.0              0
45961    CVE-2024-2973        10.0              0
41970   CVE-2024-22476        10.0              0
84900   CVE-2025-32291        10.0              0

=== TOP 10 SEGÚN MODELO ===
               cve_id  cvss_score  tiene_exploit
14967   CVE-2023-3710         9.9              1
9202   CVE-2023-36355         9.9              1
6651   CVE-2023-27823         9.8              1
6650    CVE-2023-1934         9.8              1
1140   CVE-2023-23163         9.8              1
13232  CVE-2023-39115         9.8              1
20153   CVE-2023-6019         9.8              1
312

## Guardar dataset final

In [18]:
# Guarda el dataset final
df.to_csv('../results/dataset_final.csv', index=False)

print("Guardado correctamente")

Guardado correctamente
